# Pedestrians in Kiel's Holstenstraße

**Preparation notebook for the course _Introduction to Data Science and Machine Learning_**

Holstenstraße is Kiel's main shopping street. A sensor counts the pedestrians passing by every day. This notebook loads these counts (May 2020 to January 2022) and shows them in four charts.

**What you do with it:**

1. Run the notebook from top to bottom (*Run All*). If it runs without errors, your setup works.
2. Ask your coding agent to explain each code cell to you step by step, e.g. *"Explain what this cell does and why."* You do not need to be able to write this code yourself.
3. Answer the two questions at the end of the notebook. We will discuss your answers in the first session.

## 1. Load the data

The data is stored as a CSV file in the `data/` folder of this repository. The values are separated by semicolons (`;`).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DATA_FILE = Path("..") / "data" / "pedestrians_kiel" / "holstenstrasse_2020-05-05_2022-01-03_day.csv"

pedestrians = pd.read_csv(DATA_FILE, sep=";", encoding="utf-8-sig")

# Keep the columns we need and give them short names
pedestrians = pedestrians.rename(columns={
    "time of measurement": "date",
    "pedestrians count": "count",
    "temperature in ºc": "temperature",
    "weather condition": "weather",
})[["date", "weekday", "count", "temperature", "weather"]]

# Convert the text to a real date (we only need the day, not the time zone)
pedestrians["date"] = pd.to_datetime(pedestrians["date"].str[:10])

pedestrians.head()

Before looking at charts, check whether the data looks plausible: How many days are there, which period is covered, and are the counts in a sensible range?

In [ ]:
print("Number of days:", len(pedestrians))
print("Period:", pedestrians["date"].min().date(), "to", pedestrians["date"].max().date())
pedestrians["count"].describe().round()

## 2. Pedestrians per day over time

The grey line shows the daily counts, the dark line the average of the last 7 days (a *moving average*), which smooths out the weekly ups and downs.

In [ ]:
daily = pedestrians.set_index("date")["count"]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(daily.index, daily, color="lightgrey", label="daily count")
ax.plot(daily.index, daily.rolling(7).mean(), color="black", label="7-day moving average")
ax.set_title("Pedestrians per day in Kiel's Holstenstraße")
ax.set_ylabel("pedestrians per day")
ax.legend()
plt.show()

## 3. Average pedestrians per weekday

Weekdays are sorted from Monday to Sunday (not alphabetically).

In [ ]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
per_weekday = pedestrians.groupby("weekday")["count"].mean().reindex(weekday_order)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(per_weekday.index, per_weekday)
ax.set_title("Average pedestrians per weekday")
ax.set_ylabel("pedestrians per day")
plt.show()

## 4. Average pedestrians per month

In [ ]:
per_month = daily.resample("MS").mean()

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(per_month.index.strftime("%Y-%m"), per_month)
ax.set_title("Average pedestrians per day, by month")
ax.set_ylabel("pedestrians per day")
ax.tick_params(axis="x", rotation=90)
plt.show()

## 5. Pedestrians and temperature

Each dot is one day. Sundays are shown separately, because most shops are closed.

In [ ]:
sunday = pedestrians["weekday"] == "Sunday"

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(pedestrians.loc[~sunday, "temperature"], pedestrians.loc[~sunday, "count"], alpha=0.5, label="Monday to Saturday")
ax.scatter(pedestrians.loc[sunday, "temperature"], pedestrians.loc[sunday, "count"], alpha=0.5, label="Sunday")
ax.set_title("Pedestrians per day vs. temperature")
ax.set_xlabel("temperature (°C)")
ax.set_ylabel("pedestrians per day")
ax.legend()
plt.show()

## Your turn

Write down your answers (here in this cell or on paper) and bring them to the first session:

1. **One pattern you notice in the data.** What stands out in the charts? Can you think of a reason for it? *(Tip: think about the calendar of 2020 and 2021.)*
2. **One question a shop owner in Holstenstraße might ask** – for example the owner of a bakery. Could this data answer the question, or would you need additional data?

*Optional:* ask your coding agent to add one more chart that helps to answer your question.